# Chapter 12 — Agentic Web Search

Hands-On LangChain. Code targets **LangChain 1.x**.

LLMs are trained on vast but static datasets — they can't tell you the score of last night's game,
the current weather, or the latest news. To do real-world tasks they need **live information**, usually
via web search. But the way a human browses the web is very different from how an AI agent needs to
consume information, and that distinction is the heart of **agentic search**.

In this chapter we contrast a traditional **search-and-scrape** approach with a modern **agentic
search** workflow, using one running example: *what is the weather in San Francisco, and is it a good
day to travel?*

Companion notebook for Chapter 12. Set `MODEL_PROVIDER` to `ollama` or `groq` in `.env`; Groq chat requires `GROQ_API_KEY`.

```bash
pip install ddgs requests beautifulsoup4 langchain-ollama langchain-groq ollama groq python-dotenv -q
```

> The original article uses **Tavily**, a hosted agentic-search API, for the second half. To keep this
> chapter runnable without an API key, we reach the same goal — a clean, structured, machine-readable
> answer — by pairing a free web search with the LLM's own `with_structured_output` (Chapter 2).

## 12.1 Regular search and web scraping

Let's first try to solve the weather problem like a basic program might: find a relevant webpage and
scrape it for information.

**Step 1 — find a relevant URL.** We use `ddgs` (the DuckDuckGo search library) to submit a query and
get back a list of URLs.

In [1]:
from ddgs import DDGS
import requests, re
from bs4 import BeautifulSoup

city = "San Francisco"

def search(query, max_results=6):
    """Return a list of result URLs for a query, using DuckDuckGo."""
    with DDGS() as ddgs:
        return [r["href"] for r in ddgs.text(query, max_results=max_results) if r.get("href")]

urls = search(f"current weather {city}")
for i, u in enumerate(urls, 1):
    print(i, u)

1 https://www.weather.gov/mtr/
2 https://www.localconditions.com/weather-south-san-francisco-california/94080/
3 https://www.sfgate.com/weather/
4 https://www.foxweather.com/local-weather/california/san-francisco
5 https://meteum.ai/weather/en/san-francisco
6 https://www.weather-us.com/en/california-usa/san-francisco


**Step 2 — scrape the page.** We take the first URL, download its HTML with `requests`, and pull the
text out of the heading and paragraph tags with BeautifulSoup. This is *web scraping* — extracting
information directly from a page's structure.

In [2]:
def scrape_text(url):
    try:
        resp = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=10)
        resp.raise_for_status()
        soup = BeautifulSoup(resp.text, "html.parser")
        text = "\n".join(t.get_text(" ", strip=True) for t in soup.find_all(["h1", "h2", "h3", "p"]))
        return re.sub(r"\s+", " ", text)
    except Exception as e:
        return f"[scrape failed: {type(e).__name__}: {e}]"

url = urls[0]
print("Website:", url, "\n")
print(scrape_text(url)[:600])

Website: https://www.weather.gov/mtr/ 



NWS All NOAA Areas of Severe Thunderstorms and Excessive Rainfall Today Scattered severe thunderstorms capable of damaging wind gusts are expected across the Mid-Atlantic vicinity this afternoon and evening. Additional strong to severe storms are expected across the upper Ohio Valley and Lower Great Lakes area. Excessive rainfall from heavy thunderstorms may produce areas of flash flooding from southern New Mexico into the western and northern Gulf Coast into Tuesday. Read More > Privacy Policy San Francisco Bay Area, CA Weather Forecast Office NWS Forecast Office San Francisco, CA Last Map Up


The output is cleaner than raw HTML, but it is still an unstructured block of text — and often not
even about the weather. An AI would have to do complex parsing to find the temperature, humidity, and
wind. Worse, this is **brittle**: if the site changes its layout (or returns something unexpected, as
search engines often do), the scraper breaks. The data is not in a form a machine can reliably use.

## 12.2 Agentic search

This is where **agentic search** changes the game. Agentic search is designed for AI: instead of
returning a list of links, it understands the query, gathers information from multiple sources, and
returns a **structured, data-first answer** that an agent can act on directly.

The original article does this with Tavily. We'll get the same structured outcome with no API key by
**combining a free web search with the LLM's structuring ability**: search for context, then ask the
model to return a typed object via `with_structured_output`.

**Step 1 — define the structure we want.**

In [3]:
from pydantic import BaseModel, Field
from typing import Optional

class WeatherReport(BaseModel):
    """A structured, machine-readable weather answer for an agent to act on."""
    location: str = Field(description="City and region")
    temperature_c: Optional[float] = Field(default=None, description="Current temperature in Celsius")
    conditions: str = Field(description="Short description, e.g. 'Partly cloudy', 'Mist'")
    wind_kph: Optional[float] = Field(default=None, description="Wind speed in km/h if available")
    travel_recommendation: str = Field(description="One sentence on whether it's a good day to travel")

**Step 2 — gather context with a free search, then ask the model for a structured answer.**

In [ ]:
from model_provider import get_chat_model

# gather a few search snippets (free, no key)
with DDGS() as ddgs:
    snippets = [r["body"] for r in ddgs.text(f"current weather {city} today", max_results=5)]
context = chr(10).join(snippets)

llm = get_chat_model(temperature=0)
report = llm.with_structured_output(WeatherReport).invoke(
    f"From these web search snippets about the weather in {city}, produce a structured weather "
    f"report and a travel recommendation.{chr(10) * 2}Snippets:{chr(10)}{context}"
)
report

Just like a hosted agentic-search tool, we now have a clean, unambiguous object — not a wall of text. The agent can use the fields directly, with no brittle parsing:

In [ ]:
print("Location      :", report.location)
print("Temperature C :", report.temperature_c)
print("Conditions    :", report.conditions)
print("Wind kph      :", report.wind_kph)
print("Travel advice :", report.travel_recommendation)

The difference comes down to the end-user. A human searching for weather wants a visually appealing
summary card — clarity and presentation over raw data. An AI agent needs the opposite: **structured,
predictable, machine-readable** data. It doesn't care about colors or layout; it cares about keys and
values.

By giving your agent structured search results — whether from a dedicated agentic-search API or, as
here, from a search-plus-structuring step — you build more robust applications: fewer scraping errors,
better handling of complex queries, and information in a form that lets the agent act intelligently.